# BlueSky Turkish Political Feed Analysis
**ENS491 — Graduation Project**

This notebook runs the full 5-phase pipeline sequentially.  
Each section shows the input/output files and a sample of the result.

---
**Phases:**
1. Account Verification
2. Post Collection
3. Keyword Extraction
4. Weekly Bluesky Search
5. Sentiment & Hate Speech Analysis
6. Network Analysis
7. Visualizations

## 0. Setup & Imports

In [ ]:
import subprocess, json, os
import pandas as pd

def run_script(script_path: str) -> None:
    """Run a src/ script and print its stdout/stderr."""
    result = subprocess.run(
        ["python", script_path],
        capture_output=True, text=True, encoding="utf-8"
    )
    print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr)
    return result.returncode

os.makedirs("outputs/figures", exist_ok=True)
print("Setup complete.")

---
## 1. Account Verification
**Input:** `data/combined_users_with_bsky_final.csv`  
**Output:** `outputs/verified_accounts.csv`

Calls the AT Protocol public API for each `bsky_handle` to verify it exists
and retrieve the canonical DID.

In [ ]:
run_script("src/01_verify_accounts.py")

In [ ]:
verified = pd.read_csv("outputs/verified_accounts.csv", encoding="utf-8-sig")
print(f"Total verified: {verified['verified'].sum()} / {len(verified)}")
verified.head()

---
## 2. Post Collection
**Input:** `outputs/verified_accounts.csv`  
**Output:** `outputs/all_posts_raw.jsonl`

Fetches every post for each verified account using cursor-based pagination.
Supports resumption if interrupted.

In [ ]:
run_script("src/02_fetch_posts.py")

In [ ]:
# Show first 5 posts as a DataFrame sample
posts_sample = []
with open("outputs/all_posts_raw.jsonl", "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 5:
            break
        posts_sample.append(json.loads(line))

pd.DataFrame(posts_sample)[["author_handle", "party", "created_at", "text"]].head()

---
## 3. Keyword Extraction
**Input:** `outputs/all_posts_raw.jsonl`  
**Output:** `outputs/political_keywords.json`

Combines TF-IDF over collected posts with a curated seed list of
Turkish political terms. Produces a ranked top-200 keyword list.

In [ ]:
run_script("src/03_keyword_extraction.py")

In [ ]:
with open("outputs/political_keywords.json", "r", encoding="utf-8") as f:
    kw_data = json.load(f)

print(f"Total keywords: {len(kw_data['keywords'])}")
print(f"Seed keywords: {len(kw_data['seed_keywords'])}")
print(f"TF-IDF keywords: {len(kw_data['tfidf_keywords'])}")
print("\nTop 20 keywords:")
print(kw_data["keywords"][:20])

---
## 4. Weekly Bluesky Search
**Input:** `outputs/political_keywords.json`, `outputs/verified_accounts.csv`  
**Output:** `outputs/weekly_search_results.jsonl`, `outputs/weekly_distribution_stats.json`

Searches Bluesky for each keyword in the last 7 days.  
Tags results with party info if the author is a known political actor.

In [ ]:
run_script("src/04_weekly_search.py")

In [ ]:
with open("outputs/weekly_distribution_stats.json", "r", encoding="utf-8") as f:
    stats = json.load(f)

print(f"Total posts collected: {stats['total_posts']}")
print(f"Unique authors: {stats['unique_authors']}")
print("\nTop 10 keywords by post count:")
for kw, cnt in list(stats["by_keyword"].items())[:10]:
    print(f"  {kw}: {cnt}")

---
## 5. Sentiment & Hate Speech Analysis
**Input:** `outputs/all_posts_raw.jsonl`, `outputs/weekly_search_results.jsonl`  
**Output:** `outputs/sentiment_results.csv`, `outputs/sentiment_stats.json`

Runs two TurkishBERTweet LoRA models:
- **Lora-SA**: negative / neutral / positive
- **Lora-HS**: hate speech Yes / No

> **Prerequisite:** `git clone https://github.com/ViralLab/TurkishBERTweet.git`

In [ ]:
run_script("src/05_sentiment_analysis.py")

In [ ]:
sentiment_df = pd.read_csv("outputs/sentiment_results.csv", encoding="utf-8-sig")
print(f"Total rows: {len(sentiment_df)}")
print("\nSentiment distribution:")
print(sentiment_df["sentiment"].value_counts())
print("\nHate speech distribution:")
print(sentiment_df["hate_speech"].value_counts())
sentiment_df[["author_handle", "party", "sentiment", "hate_speech", "text_preview"]].head()

---
## 6. Network Analysis
**Input:** `outputs/all_posts_raw.jsonl`, `outputs/verified_accounts.csv`  
**Output:** `outputs/network_edges.csv`, `outputs/network_metrics.json`

Builds a directed interaction graph (reply + quote edges),  
computes centrality metrics and detects communities.

In [ ]:
run_script("src/06_network_analysis.py")

In [ ]:
edges_df = pd.read_csv("outputs/network_edges.csv", encoding="utf-8-sig")
print(f"Total edges: {len(edges_df)}")

with open("outputs/network_metrics.json", "r", encoding="utf-8") as f:
    metrics = json.load(f)

print(f"Nodes: {metrics['total_nodes']}, Edges: {metrics['total_edges']}")
print(f"Communities detected: {metrics['num_communities']}")
print(f"Intra-party ratio: {metrics['intra_party_ratio']:.1%}")
print("\nTop 5 most-mentioned accounts (in-degree):")
for entry in metrics["top_20_by_in_degree"][:5]:
    print(f"  {entry['handle']}: {entry['in_degree']}")

---
## 7. Visualizations
**Input:** all outputs from previous steps  
**Output:** `outputs/figures/*.png` and `*.html`

Generates 9 figures:
- G1: Party account & post counts
- G2: Weekly post volume time series
- G3: Sentiment distribution per party
- G4: Hate speech rate per party
- G5: Cross-party sentiment heatmap
- G6: Interactive network graph (HTML)
- G7: Top 20 most active accounts
- G8: Per-party word clouds
- G9: Party interaction Sankey diagram (HTML)

In [ ]:
run_script("src/07_visualizations.py")

In [ ]:
# Display all generated PNG figures inline
from IPython.display import Image, display
import glob

pngs = sorted(glob.glob("outputs/figures/*.png"))
print(f"Generated {len(pngs)} PNG figures.")
for path in pngs:
    print(f"\n{'─'*50}")
    print(os.path.basename(path))
    display(Image(filename=path, width=900))

---
## 8. Summary Statistics & Key Findings

In [ ]:
# Compile a high-level summary across all pipeline outputs
summary_lines = []

if os.path.exists("outputs/verified_accounts.csv"):
    va = pd.read_csv("outputs/verified_accounts.csv", encoding="utf-8-sig")
    summary_lines.append(f"Verified accounts: {va['verified'].sum()} of {len(va)}")

if os.path.exists("outputs/all_posts_raw.jsonl"):
    n_posts = sum(1 for _ in open("outputs/all_posts_raw.jsonl", encoding="utf-8"))
    summary_lines.append(f"Actor posts collected: {n_posts:,}")

if os.path.exists("outputs/weekly_distribution_stats.json"):
    with open("outputs/weekly_distribution_stats.json", encoding="utf-8") as f:
        ws = json.load(f)
    summary_lines.append(f"Weekly search posts: {ws['total_posts']:,}")
    summary_lines.append(f"Weekly unique authors: {ws['unique_authors']:,}")

if os.path.exists("outputs/sentiment_results.csv"):
    sd = pd.read_csv("outputs/sentiment_results.csv", encoding="utf-8-sig")
    dist = sd["sentiment"].value_counts(normalize=True).round(3).to_dict()
    summary_lines.append(f"Sentiment distribution: {dist}")
    hs_rate = (sd["hate_speech"] == "Yes").mean()
    summary_lines.append(f"Overall hate speech rate: {hs_rate:.1%}")

if os.path.exists("outputs/network_metrics.json"):
    with open("outputs/network_metrics.json", encoding="utf-8") as f:
        nm = json.load(f)
    summary_lines.append(f"Network nodes: {nm['total_nodes']}, edges: {nm['total_edges']}")
    summary_lines.append(f"Intra-party interaction ratio: {nm['intra_party_ratio']:.1%}")

print("=" * 55)
print("  PIPELINE SUMMARY")
print("=" * 55)
for line in summary_lines:
    print(f"  • {line}")